# A fairer few-shot baseline

Baselines.ipynb found that no prompt gets close to the adapter on style (few-shot 3B: style position 0.32 TF-IDF / 0.59 mDeBERTa, against 0.96 / 0.81). A reviewer's objection: that baseline got one untuned prompt and no length limit, and untrained models write long replies, which the classifiers partly pick up (a length-only classifier already puts few-shot at 0.04 to 0.10). Here the prompting baseline gets a fair chance:

- Qwen2.5-0.5B, 1.5B and 3B, untrained (the sizes of Baselines.ipynb; the 7B is only the judge), with the same 20 real replies of mine from train in the system prompt (`--few-shot 20`), temperature 1.0.
- Two instructions appended to the system prompt: **short** ("one short WhatsApp message, one sentence, like the examples") and **short + style** (also: informal, lowercase, no final full stop, as in the examples).
- A hard cap of 64 tokens per reply (my replies: median 13, mean 17.5).
- Style with the same frozen classifiers (length, TF-IDF, mDeBERTa) and relevance with the same Qwen2.5-7B judge as Baselines.ipynb.

Runs on the NAS GPU (generation, classifiers and judge).

## Prediction (written before running)

Fede: no way it reaches the adapter. Style goes up against the original few-shot baseline (that's the idea), but doesn't reach the adapter (0.96 TF-IDF / 0.81 mDeBERTa).

Claude: style goes up a lot with TF-IDF, because the instruction targets what it looks at (punctuation, lowercase, length), less with mDeBERTa, and none reaches the adapter. Relevance doesn't improve, or drops a little, because a forced short sentence says less.

How it's read: "reaches the adapter" means a style position whose interval overlaps the adapter's on both classifiers.

Note: the "short + style" instruction describes features a reader would notice in my examples (informal, lowercase, no final full stop); some of them are also what the TF-IDF classifier relies on.

In [1]:
import json, os, subprocess
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
PY = os.environ.get("TLM_PY", "uv run python")
JUDGE = "mlx-community/Qwen2.5-7B-Instruct-4bit"
MODELS = {"0.5b": "mlx-community/Qwen2.5-0.5B-Instruct-4bit", "1.5b": "mlx-community/Qwen2.5-1.5B-Instruct-4bit", "3b": "mlx-community/Qwen2.5-3B-Instruct-4bit"}
INSTRUCTIONS = {
    "short": "Responde con un solo mensaje corto de WhatsApp, de una frase, como en los ejemplos.",
    "style": "Responde con un solo mensaje corto de WhatsApp, de una frase, como en los ejemplos. "
             "Escribe informal, en minúsculas y sin punto final, igual que en los ejemplos.",
}
NAMES = [f"fewshot20_{m}_{i}" for m in MODELS for i in INSTRUCTIONS]

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line and "█" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

## 1. Generate (temperature 1.0, at most 64 tokens)

In [2]:
for m, model in MODELS.items():
    for i, text in INSTRUCTIONS.items():
        run(f'{PY} scripts/generate.py --name fewshot20_{m}_{i} --model {model} --few-shot 20 --max-tokens 64 --instruction "{text}"')

$ /opt/venv/bin/python scripts/generate.py --name fewshot20_0.5b_short --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --few-shot 20 --max-tokens 64 --instruction "Responde con un solo mensaje corto de WhatsApp, de una frase, como en los ejemplos."
100/1133 (0.5 min)
200/1133 (0.8 min)
300/1133 (1.1 min)
400/1133 (1.3 min)
500/1133 (1.6 min)
600/1133 (1.9 min)
700/1133 (2.1 min)
800/1133 (2.4 min)
900/1133 (2.7 min)
1000/1133 (2.9 min)
1100/1133 (3.2 min)
1133 replies in data/style/generations_fewshot20_0.5b_short.jsonl
$ /opt/venv/bin/python scripts/generate.py --name fewshot20_0.5b_style --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --few-shot 20 --max-tokens 64 --instruction "Responde con un solo mensaje corto de WhatsApp, de una frase, como en los ejemplos. Escribe informal, en minúsculas y sin punto final, igual que en los ejemplos."
100/1133 (0.3 min)
200/1133 (0.6 min)
300/1133 (0.8 min)
400/1133 (1.1 min)
500/1133 (1.4 min)
600/1133 (1.6 min)
700/1133 (1.9 min)
800/1133 (2.

## 2. Style: the same frozen classifiers

In [3]:
if not (R / "style_fewshot_fair.json").exists():
    run(f"{PY} scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra fewshot20_0.5b fewshot20_1.5b fewshot20_3b {' '.join(NAMES)} --transformer --out experiments/results/style_fewshot_fair.json")

$ /opt/venv/bin/python scripts/eval_style.py --adapter-gens adapter_t1 --base-gens base_t1 --extra fewshot20_0.5b fewshot20_1.5b fewshot20_3b fewshot20_0.5b_short fewshot20_0.5b_style fewshot20_1.5b_short fewshot20_1.5b_style fewshot20_3b_short fewshot20_3b_style --transformer --out experiments/results/style_fewshot_fair.json
length (test AUC on real messages 0.551)
  base       classified as me   7.8% [5.6%, 9.5%]
  adapter    classified as me  41.9% [39.6%, 44.4%]
  fewshot20_0.5b classified as me   9.3% [6.7%, 11.1%]
  fewshot20_1.5b classified as me  11.8% [9.2%, 14.0%]
  fewshot20_3b classified as me  11.6% [9.0%, 13.2%]
  fewshot20_0.5b_short classified as me  13.5% [10.7%, 16.3%]
  fewshot20_0.5b_style classified as me  14.4% [12.4%, 16.3%]
  fewshot20_1.5b_short classified as me  16.3% [13.3%, 18.4%]
  fewshot20_1.5b_style classified as me  18.1% [15.3%, 21.1%]
  fewshot20_3b_short classified as me  38.7% [34.4%, 42.0%]
  fewshot20_3b_style classified as me  46.7% [42.4%, 50.9%

## 3. Relevance, judged by Qwen2.5-7B (same cache as Baselines.ipynb)

In [4]:
if not (R / "relevance_fewshot_fair.json").exists():
    run(f"{PY} scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra fewshot20_0.5b fewshot20_1.5b fewshot20_3b {' '.join(NAMES)} "
        f"--scorer {JUDGE} --cache experiments/results/relevance_nll_7b.json --out experiments/results/relevance_fewshot_fair.json")

$ /opt/venv/bin/python scripts/eval_relevance.py --adapter-gens adapter_t1 --base-gens base_t1 --extra fewshot20_0.5b fewshot20_1.5b fewshot20_3b fewshot20_0.5b_short fewshot20_0.5b_style fewshot20_1.5b_short fewshot20_1.5b_style fewshot20_3b_short fewshot20_3b_style --scorer mlx-community/Qwen2.5-7B-Instruct-4bit --cache experiments/results/relevance_nll_7b.json --out experiments/results/relevance_fewshot_fair.json
fewshot20_0.5b_short: 100/1133 (0.8 min)
fewshot20_0.5b_short: 200/1133 (1.0 min)
fewshot20_0.5b_short: 300/1133 (1.3 min)
fewshot20_0.5b_short: 400/1133 (1.5 min)
fewshot20_0.5b_short: 500/1133 (1.8 min)
fewshot20_0.5b_short: 600/1133 (2.0 min)
fewshot20_0.5b_short: 700/1133 (2.2 min)
fewshot20_0.5b_short: 800/1133 (2.5 min)
fewshot20_0.5b_short: 900/1133 (2.7 min)
fewshot20_0.5b_short: 1000/1133 (3.0 min)
fewshot20_0.5b_short: 1100/1133 (3.2 min)
fewshot20_0.5b_short: 1133/1133 (3.3 min)
fewshot20_0.5b_style: 100/1133 (3.5 min)
fewshot20_0.5b_style: 200/1133 (3.8 min)
few

## Summary

In [5]:
style = json.load(open(R / "style_fewshot_fair.json"))["scores"]
rel = json.load(open(R / "relevance_fewshot_fair.json"))["positions"]
print(f"{'set':<24} {'length':>7} {'TF-IDF':>7} {'mDeBERTa':>9} {'relevance':>10}")
for k in ["adapter", "fewshot20_0.5b", "fewshot20_1.5b", "fewshot20_3b"] + NAMES:
    r = rel["adapter t1.0" if k == "adapter" else k]
    print(f"{k:<24} " + " ".join(f"{style[c][f'position {k}']['value']:>7.2f}" for c in ["length", "tfidf"])
          + f" {style['transformer'][f'position {k}']['value']:>9.2f} {r['value']:>10.2f}")
print("positions: style 0 = base, 1 = me; relevance 0 = my reply to another chat, 1 = my real reply")

set                       length  TF-IDF  mDeBERTa  relevance
adapter                     0.86    0.96      0.81       0.27
fewshot20_0.5b              0.04   -0.02     -0.25      -0.55
fewshot20_1.5b              0.10    0.47      0.32      -0.11
fewshot20_3b                0.10    0.32      0.59       0.27
fewshot20_0.5b_short        0.14    0.19     -0.21      -0.39
fewshot20_0.5b_style        0.17    0.18     -0.14      -0.34
fewshot20_1.5b_short        0.22    0.49      0.54       0.30
fewshot20_1.5b_style        0.26    0.47      0.40       0.29
fewshot20_3b_short          0.78    0.26      0.50       1.27
fewshot20_3b_style          0.98    0.22      0.25       1.29
positions: style 0 = base, 1 = me; relevance 0 = my reply to another chat, 1 = my real reply


## Conclusion

**Prediction:** Fede, style goes up against the original few-shot but doesn't reach the adapter; Claude, style goes up a lot with TF-IDF and less with mDeBERTa, none reaches the adapter, and relevance doesn't improve. **Fede right; Claude wrong on both counts. And relevance did something nobody expected.**

| Set | Style TF-IDF | Style mDeBERTa | Relevance position (CI) | Context gain (nats) | Mean characters |
|---|---|---|---|---|---|
| Fine-tuned 0.5B | 0.96 | 0.81 | 0.27 [0.09, 0.42] | 0.336 | 52 |
| My real replies | 1 | 1 | 1 | 0.658 | 48 |
| Few-shot 3B, original | 0.32 | 0.59 | 0.27 [0.07, 0.46] | 0.335 | 141 |
| Few-shot 3B, short | 0.26 | 0.50 | **1.27 [1.06, 1.52]** | 0.780 | 42 |
| Few-shot 3B, short + style | 0.22 | 0.25 | **1.29 [1.07, 1.52]** | 0.788 | 38 |
| Few-shot 1.5B, short / + style | 0.49 / 0.47 | 0.54 / 0.40 | 0.30 / 0.29 | 0.349 / 0.343 | 72 / 71 |
| Few-shot 0.5B, short / + style | 0.19 / 0.18 | -0.21 / -0.14 | -0.39 / -0.34 | 0.043 / 0.064 | 113 / 109 |

- **Style: a fair prompt doesn't get close to the adapter.** The instructions barely move style: the 0.5B goes up a little with TF-IDF (-0.02 to 0.19), the 1.5B stays about the same, and the 3B goes *down* (0.32 to 0.22-0.26 with TF-IDF, 0.59 to 0.25-0.50 with mDeBERTa) even though its replies are now as short as mine. So length was not what kept few-shot away from my style; asking for "lowercase, no full stop" doesn't produce it either. Training is still the only method here that gets the style.
- **Relevance: the short 3B scores above my own replies.** Capped and told to be brief, the 3B's replies get a context gain of 0.78 nats, more than my real replies (0.66): position 1.27, interval clear of 1. They are not copies of the conversation (0.3% of replies are contained in it, 0.1% are near-copies of the last message). The likely reading: a short, on-topic answer from an instruction-tuned model is exactly what a model judge finds most predictable from the conversation, more than my real replies, which refer to things the judge can't know. Untested alternative: the judge (Qwen2.5-7B) shares a family with the 3B and prefers its way of writing.

**What it changes.** The relevance measure rewards "predictable from the conversation", and a generic but on-topic reply can beat the real person on it. It still separates my real reply from a reply of mine to another conversation (decision 20), but it is not a measure of "what I would say", and claims based on it are weaker than they looked: the adapter's low relevance may partly be that its replies, like mine, are idiosyncratic. Next check: a judge from another model family, to see whether the 3B still scores above my real replies.

**Limits.** One sample per context; one prompt per instruction; the "short + style" instruction names features a reader sees in my examples, some of which the TF-IDF classifier uses.